<a href="https://colab.research.google.com/github/TinoTonic/REDCap-SDTM-automation/blob/main/REDCap_SDTM_Bridge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Required Input Files

The notebook requires the following files:

- REDCap data dictionary in .csv

- SDTM controlled terminology (`SDTM Terminology 2026-03-27`)

- SDTMIG (`SDTMIG_v3.4.xlsx`)

If you want to generate an annotated CRF:

- Annotated CRF PDF generated by the external module `Annotated PDF`.



## 1. Installing and importing packages

In [ ]:
pip install thefuzz pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 87.4 MB/s eta 0:00:00


In [ ]:
import re
import pandas as pd
import pymupdf as fitz
from thefuzz import process
from pymupdf.utils import getColorList
from pymupdf.utils import getColorInfoList
il = getColorInfoList()
cl = getColorList()

## 2. Reading the Data Dictionary

In [ ]:
redcap_data = pd.read_csv("redcap_data.csv")

redcap_data = redcap_data.rename(columns={
    "Variable / Field Name": "Field Name",
    "Choices, Calculations, OR Slider Labels": "Choices"
     })
redcap_data = redcap_data.drop(columns=[
    'Question Number (surveys only)',
    'Section Header',
    'Field Note',
    'Text Validation Min',
    'Text Validation Max',
    'Branching Logic (Show field only if...)',
    'Custom Alignment',
    'Question Number (surveys only)',
    'Matrix Group Name',
    'Matrix Ranking?'
    ])

## 3. Reading REDCap Field Annotations

Looking for

- SDTM:IT.domain.variable.testcd;

And

- Y, Yes | N, No (example)

In the rows "Field Annotation" and "Choices" respectively.





In [ ]:
sdtm_dataset = pd.read_excel("SDTMIG_v3.4.xlsx", sheet_name = "Datasets")

redcap_data = redcap_data.copy()
rawCL = []
domains = []

sdtm_pattern = r"SDTM[.,:]IT[.,:](?P<DOMAIN>\w+)\.(?P<SDTM_VARIABLE>\w+)(?:\.(?P<TESTCD>\w+))?[;.:]?"
submission_pattern = r"\s*(?P<REDCap_Value>[^,|]+)\s*,\s*(?P<REDCap_Term>[^|]+)"

for _, row in redcap_data.iterrows():
    field_annotation = str(row["Field Annotation"]).strip()
    field_name = str(row["Field Name"]).strip()
    values = str(row["Choices"]).strip()
    field_type = str(row["Field Type"]).strip()
    if field_type == "sql":
        continue
    if field_name.startswith("desc_"):
        redcap_data.at[_, "Field Label"] = ""
    match = re.search(sdtm_pattern, field_annotation)
    if match:
        redcap_data.at[_, "DOMAIN"] = match.group("DOMAIN")
        redcap_data.at[_, "SDTM VARIABLE"] = match.group("SDTM_VARIABLE")
        redcap_data.at[_, "TESTCD"] = match.group("TESTCD")
    else:
        redcap_data.at[_, "DOMAIN"] = ""
        redcap_data.at[_, "SDTM VARIABLE"] = ""
        redcap_data.at[_, "TESTCD"] = ""
    if redcap_data.at[_, "DOMAIN"] != "":
        domains.append({
        "Domain": redcap_data.at[_, "DOMAIN"]
        })

    for match in re.finditer(submission_pattern, values):
          rawCL.append({
              "Field Name": field_name,
              "REDCap Value": match.group("REDCap_Value"),
              "REDCap Term": match.group("REDCap_Term").strip()
          })

domains = pd.DataFrame(domains)
domains = domains.drop_duplicates()
supp_list = [dom[-2:] for dom in domains["Domain"] if dom.startswith("SUPP")]

datasetmetadata = (domains.merge(
        sdtm_dataset,
        left_on="Domain",
        right_on="Dataset Name",
        how="inner"
        )
        [["Dataset Name","Class","Dataset Label","Structure"]]
)

supp_rows_data = []
for dom in supp_list:
    new_row = {
        "Dataset Name": f"SUPP{dom}",
        "Class": "Relationship",
        "Dataset Label": f"Supplemental Qualifiers for {dom}",
        "Structure": "One record per IDVAR, IDVARVAL, and QNAM value per subject",
        }
    supp_rows_data.append(new_row)
df_supp_metadata = pd.DataFrame(supp_rows_data)
datasetmetadata = pd.concat([datasetmetadata, df_supp_metadata], ignore_index=True)

rawcodelist = pd.DataFrame(rawCL)

## 4. Reading files



In [ ]:
mapping = redcap_data.copy()
mapping.fillna("")

codelist = rawcodelist.copy()
codelist.fillna("")

terminology = pd.read_excel("SDTM Terminology.xls", sheet_name="SDTM Terminology 2026-03-27")

sdtm_variables = pd.read_excel("SDTMIG_v3.4.xlsx", sheet_name = "Variables")

##5. Filtering and generating codelists of every mapped variable

In [ ]:
mapping["SDTM VARIABLE"] = (mapping["SDTM VARIABLE"].astype(str).str.strip())

mapping_filtered = mapping[(mapping["SDTM VARIABLE"] != "") & (mapping["SDTM VARIABLE"] != "nan")]
mapping_filtered = pd.DataFrame(mapping_filtered)

mapping_filtered.loc[mapping_filtered["SDTM VARIABLE"] == "RACE", "SDTM VARIABLE"] = "RACEC"

MyCl = (mapping_filtered.merge(
        codelist,
        left_on="Field Name",
        right_on="Field Name",
        how="inner"
        )
        [["Field Name","SDTM VARIABLE","TESTCD","REDCap Term","REDCap Value"]]
)

MyCodelist = MyCl.to_dict("records")
MyCodelist = pd.DataFrame(MyCodelist)

##6. Linking mapped variables to their respective codelists

In [ ]:
link = {}
linked_by_code = []

for _, row in mapping.iterrows():
    field_name = str(row["Field Name"]).strip()
    domain = str(row["DOMAIN"]).strip()
    variable = str(row["SDTM VARIABLE"]).strip()
    testcd = str(row["TESTCD"]).strip()
    if not variable:
        continue

    for _, row in sdtm_variables.iterrows():
        var = str(row["Variable Name"]).strip()
        if var == variable:
            codelistcode = str(row["CDISC CT Codelist Code(s)"]).strip()
            if codelistcode == "nan":
                continue
            link[variable] = {"Variable": variable,"Codelist Code": codelistcode}

df = pd.DataFrame(link).T

for _, df_row in df.iterrows():
    variable = df_row["Variable"]
    dfclcode = df_row["Codelist Code"]

    for _, term_row in terminology.iterrows():
        clcode = str(term_row["Codelist Code"]).strip()
        subval = str(term_row["CDISC Submission Value"]).strip()

        if clcode == dfclcode:
            linked_by_code.append({
                "Variable": variable,
                "Codelist Name": str(term_row["Codelist Name"]).strip(),
                "Codelist Code": dfclcode,
                "Codes": str(term_row["Code"]).strip(),
                "Submission Value": str(term_row["CDISC Submission Value"]).strip(),
                "NCI Pref. Term": str(term_row["NCI Preferred Term"]).strip()
            })
        if subval == variable or subval == f"{variable}C":
            code = str(term_row["Code"]).strip()
            saved_vars = [item["Variable"] for item in linked_by_code]
            for _, term_row in terminology.iterrows():
                if code == str(term_row["Codelist Code"]).strip():
                    if subval not in saved_vars:
                        linked_by_code.append({
                            "Variable": subval,
                            "Codelist Name": str(term_row["Codelist Name"]).strip(),
                            "Codelist Code": str(term_row["Codelist Code"]).strip(),
                            "Codes": str(term_row["Code"]).strip(),
                            "Submission Value": str(term_row["CDISC Submission Value"]).strip(),
                            "NCI Pref. Term": str(term_row["NCI Preferred Term"]).strip()
                        })

df2 = pd.DataFrame(linked_by_code)
df2 = df2.drop_duplicates()

## 7. Generating study specific controlled terminology by similarity

In [ ]:
SDTM_Values = df2["Submission Value"].astype(str).tolist()

def get_fuzz_match(text, choices_list, threshold=50):
    match, score = process.extractOne(str(text), choices_list)
    if score >= threshold:
        return match
    else:
        return None

MyCodelist["Temp"] = MyCodelist["REDCap Term"].apply(
    lambda x: get_fuzz_match(x, SDTM_Values, threshold=70)
)

MyCodelist.loc[MyCodelist["SDTM VARIABLE"] == "QVAL", "SDTM VARIABLE"] = MyCodelist["TESTCD"]
MyCodelist.loc[MyCodelist["SDTM VARIABLE"] == "RACE", "SDTM VARIABLE"] = "RACEC"
MyCodelist.loc[MyCodelist["SDTM VARIABLE"] == "ETHNIC", "SDTM VARIABLE"] = "ETHNICC"
MyCodelist.loc[MyCodelist["SDTM VARIABLE"] == "CETHNIC", "SDTM VARIABLE"] = "ETHNICC"

CT = (
    MyCodelist.merge(
        df2,
        left_on=["SDTM VARIABLE","Temp"],
        right_on=["Variable","Submission Value"],
        how="right"
    )
    [["Field Name","SDTM VARIABLE","TESTCD","Codelist Name","Codelist Code", "Codes", "REDCap Value", "REDCap Term", "Submission Value", "NCI Pref. Term"]]
)

CT.loc[CT["SDTM VARIABLE"] == "RACEC", "SDTM VARIABLE"] = "RACE"

MyCodelist.drop(columns=["Temp"], inplace=True)

CT = CT.drop_duplicates(subset=["SDTM VARIABLE", "TESTCD", "Codelist Code", "Codes"])

## 8. Output Files

In [ ]:
with pd.ExcelWriter("REDCap Mapping SPEC.xlsx", engine="openpyxl") as writer:
    datasetmetadata.to_excel(writer, sheet_name="Dataset Metadata", index=False)
    redcap_data.to_excel(writer, sheet_name="Variable Metadata", index=False)
    rawcodelist.to_excel(writer, sheet_name="REDCap Values", index=False)
    df2.to_excel(writer, sheet_name="Mapped Variables Codelists", index=False)
    CT.to_excel(writer, sheet_name="Linked Codelist", index=False)

## Annotated CRF



##1. Reading files

In [ ]:
mapping = pd.read_excel('REDCap Mapping SPEC.xlsx', sheet_name='Variable Metadata')

dataset = pd.read_excel('REDCap Mapping SPEC.xlsx', sheet_name='Dataset Metadata')
dataset = dataset[["Dataset Name", "Dataset Label"]]
dataset = dataset[~dataset["Dataset Name"].str.startswith("SUPP")]

crf = "AnnotatedPDF.pdf"

acrf = "ACRF.pdf"

/tmp/ipykernel_20782/92705868.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  mapping.fillna("",inplace=True)


##2. Choosing domain collors



In [ ]:
il = getColorInfoList()
cl = getColorList()

In [ ]:
DEFAULT_COLOR = (0.48, 0.8, 0.48) # Light green

DOMAIN_COLORS = {
    "DM": (0.48, 0.80, 0.48),   # Light green
    "SUPPDM": (0.48, 0.80, 0.48),
    "AE": (0.98, 0.60, 0.20),   # Orange
    "SUPPAE": (0.98, 0.60, 0.20),
    "CM": (0.42, 0.71, 0.91),   # Blue
    "SUPPCM": (0.42, 0.71, 0.91),
    "MH": (0.85, 0.55, 0.85),   # Purple
    "SUPPMH": (0.85, 0.55, 0.85),
    "LB": (0.95, 0.85, 0.30),   # Yellow
    "SUPPLB": (0.95, 0.85, 0.30),
    "CE": (0.30, 0.85, 0.85),   # Light Blue
    "SUPPCE": (0.30, 0.85, 0.85),
    "FA": (0.90, 0.45, 0.45),   # Red
    "SUPPFA": (0.90, 0.45, 0.45),
    "DS": (1, 0.75, 0.79),      # Pink
    "SUPPDS": (1, 0.75, 0.79),
    "QS": (1,0.64,0.05),        # Orange
    "SUPPQS":(1,0.64,0.05)
    }

##3. Generating the aCRF

In [ ]:
doc = fitz.open(crf)

for _, row in mapping.iterrows():
    field_name = str(row["Field Name"]).strip()
    domain = str(row["DOMAIN"]).strip()
    variable = str(row["SDTM VARIABLE"]).strip()
    testcd = str(row["TESTCD"]).strip()

    if not field_name:
        continue

    for page in doc:
        matches = page.search_for("{"f"[{field_name}]")
        for rect in matches:
            if domain.startswith("SUPP"):
                box = fitz.Rect(rect.x0 - 20,rect.y1 + 1,rect.x0 + 60,rect.y1 + 10)
            elif testcd:
                box = fitz.Rect(rect.x0 - 20,rect.y1 + 1,rect.x0 + 110,rect.y1 + 10)
            else:
                box = fitz.Rect(rect.x0,rect.y1 + 1,rect.x0 + 52,rect.y1 + 10)

            if not domain:
                fieldcolor = (0.85, 0.85, 0.85)
            else:
                fieldcolor = DOMAIN_COLORS.get(domain, DEFAULT_COLOR)

            page.draw_rect(box, color=(0, 0, 0), fill = fieldcolor, overlay=True)

            if domain and variable and testcd:
                if domain.startswith("SUPP"):
                    page.insert_text((box.x0 + 2, box.y1 - 2),f"{testcd} in {domain}",fontsize=6,color=(0, 0, 0),overlay=True)
                else:
                    page.insert_text((box.x0 + 2, box.y1 - 2),f'{variable} where {domain}TESTCD = "{testcd}"',fontsize=6,color=(0, 0, 0),overlay=True)

            elif domain and variable:
                page.insert_text((box.x0 + 2, box.y1 - 2),f"{domain}.{variable}",fontsize=6,color=(0, 0, 0),overlay=True)
            else:
                page.insert_text((box.x0 + 2, box.y1 - 2),"NOT SUBMITTED",fontsize=6,color=(0, 0, 0),overlay=True)

for i in mapping.index:
    domain = str(mapping.loc[i,"DOMAIN"])
    form = str(mapping.loc[i,"Form Name"])

    if domain and form:
        form = str(form).title().replace("_", " ").strip()
        formdomain = f"{form} = ({domain})"
        mapping.loc[i,"FORM.DOMAIN"] = formdomain
        mapping.loc[i,"FORM"] = form

mapping_filtered = mapping.drop_duplicates(subset=["FORM.DOMAIN"])

alldomains = (mapping_filtered.groupby("FORM")["DOMAIN"].apply(lambda x: ", ".join(x)).to_dict())

for formname, dominios in alldomains.items():

    for page in doc:
        matches = page.search_for(f"{formname}")

        for rect in matches:
            if rect.y0 > 40:
                continue
            if rect.height<15:
                continue

            height = 14
            distance = 6
            x0box = rect.x1 + 8
            y0box = (rect.y0 + (rect.height - height) / 2) - 25
            y1box = y0box + height

            for dom in dominios.split(", "):
                if dom.startswith("SUPP"):
                    continue

                domain_search = str(dom).strip().upper()
                filter = dataset[dataset["Dataset Name"] == domain_search]["Dataset Label"]
                if not filter.empty:
                    label = filter.values[0]
                    pagetext = f"{dom} = {label}"
                else:
                    print(f'Warning: Domain "{domain_search}" is missing from "Dataset Metadata"')
                    pagetext = ""

                width = (len(pagetext)*6)
                if width >= 200:
                    width = 200

                domaincolor = DOMAIN_COLORS.get(dom, DEFAULT_COLOR)

                box = fitz.Rect(x0box, y0box, x0box + width, y1box)

                page.draw_rect(box, color=(0, 0, 0), fill=domaincolor, overlay=True)

                page.insert_text((box.x0 + 4, box.y1 - 3), pagetext, fontsize=10, color=(0, 0, 0), overlay=True)

                x0box += distance + 5
                y1box += height + 4
                y0box += height + 4

doc.save(acrf)